In [ ]:
# Make important import and create directory

import sqlite3
import os
import pandas as pd


# Define the directory and file paths
db_dir = os.path.join("..", 'database')
db_path = os.path.join(db_dir, 'crypto.db')

# Check if the directory exists, if not, create it
if not os.path.exists(db_dir):
    os.makedirs(db_dir)
    print(f"Directory '{db_dir}' created.")

# If the database file already exists, delete it
if os.path.exists(db_path):
    os.remove(db_path)
    print("Database file deleted.")
else:
    print("Database file does not exist.")


Database file deleted.


In [2]:
print(db_path)

..\database\crypto.db


In [3]:
# Create a new database connection

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create the 'coins' table
cursor.execute("""
CREATE TABLE IF NOT EXISTS coins (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    symbol TEXT NOT NULL UNIQUE
);
""")

# Create the 'prices' table with all necessary columns
cursor.execute("""
CREATE TABLE IF NOT EXISTS prices (
    date TIMESTAMP NOT NULL,
    symbol TEXT NOT NULL,
    price REAL NOT NULL,
    market_cap REAL,
    volume_24h REAL,
    FOREIGN KEY (symbol) REFERENCES coins (symbol)
);
""")
conn.commit()
print("Tables 'coins' and 'prices' created successfully.")

Tables 'coins' and 'prices' created successfully.


In [4]:
# Data for the coins table
coins_table = [
    ('Ethereum', 'ETH'),
    ('Bitcoin', 'BTC'),
    ('Ripple', 'XRP'),
    ('Litecoin', 'LTC')
]

# Insert data into the 'coins' table
try:
    cursor.executemany("""
    INSERT INTO coins (name, symbol)
    VALUES (?, ?)
    """, coins_table)
    conn.commit()
    print("Coins inserted successfully.")
except sqlite3.IntegrityError:
    print("Coins already exist in the database.")

Coins inserted successfully.


In [5]:
# Data for the prices table (matches the table schema)
daily_price_table = [
    ('2023-10-01', 'ETH', 2000.50, 200000000, 15000000),
    ('2023-10-01', 'BTC', 40000.00, 800000000, 30000000),
    ('2023-10-01', 'XRP', 0.50, 25000000, 5000000),
    ('2023-10-01', 'LTC', 150.00, 10000000, 2000000),
    ('2023-10-02', 'ETH', 2050.00, 210000000, 16000000),
    ('2023-10-02', 'BTC', 41000.00, 810000000, 31000000),
    ('2023-10-02', 'XRP', 0.55, 26000000, 5500000),
    ('2023-10-02', 'LTC', 155.00, 11000000, 2200000),
    ('2023-10-03', 'ETH', 2100.00, 220000000, 17000000),
    ('2023-10-03', 'BTC', 42000.00, 820000000, 32000000),
    ('2023-10-03', 'XRP', 0.60, 27000000, 6000000),
    ('2023-10-03', 'LTC', 160.00, 12000000, 2400000),
    ('2023-10-04', 'ETH', 2150.00, 230000000, 18000000),
    ('2023-10-04', 'BTC', 43000.00, 830000000, 33000000),
    ('2023-10-04', 'XRP', 0.65, 28000000, 6500000),
    ('2023-10-04', 'LTC', 165.00, 13000000, 2600000),
    ('2023-10-05', 'ETH', 2200.00, 240000000, 19000000),
    ('2023-10-05', 'BTC', 44000.00, 840000000, 34000000),
    ('2023-10-05', 'XRP', 0.70, 29000000, 7000000),
    ('2023-10-05', 'LTC', 170.00, 14000000, 2800000),
    ('2023-10-06', 'ETH', 2250.00, 250000000, 20000000),
    ('2023-10-06', 'BTC', 45000.00, 850000000, 35000000),
    ('2023-10-06', 'XRP', 0.75, 30000000, 7500000),
    ('2023-10-06', 'LTC', 175.00, 15000000, 3000000)
]

In [ ]:
# Insert daily prices into the prices table
cursor.executemany("""
INSERT INTO prices (date, symbol, price, market_cap, volume_24h)
VALUES (?, ?, ?, ?, ?)
""", daily_price_table)
# Save changes
try:
    conn.commit()
    print("Daily prices inserted successfully.")
except Exception as e:
    conn.rollback()
    print(f"Daily prices insertion failed: {e}")
# Close the connection
conn.close()


Daily prices inserted successfully.


In [7]:
# Join the 'coins' and 'prices' tables to get the daily prices for each coin
conn = sqlite3.connect(db_path)
query = """
SELECT c.name, c.symbol, p.date, p.price, p.market_cap, p.volume_24h
FROM coins c
JOIN prices p ON c.symbol = p.symbol
ORDER BY p.date, c.symbol;
"""
cursor = conn.cursor()
cursor.execute(query)
results = cursor.fetchall()
for row in results:
    print(row)
# Close the connection
conn.close()


('Bitcoin', 'BTC', '2023-10-01', 40000.0, 800000000.0, 30000000.0)
('Ethereum', 'ETH', '2023-10-01', 2000.5, 200000000.0, 15000000.0)
('Litecoin', 'LTC', '2023-10-01', 150.0, 10000000.0, 2000000.0)
('Ripple', 'XRP', '2023-10-01', 0.5, 25000000.0, 5000000.0)
('Bitcoin', 'BTC', '2023-10-02', 41000.0, 810000000.0, 31000000.0)
('Ethereum', 'ETH', '2023-10-02', 2050.0, 210000000.0, 16000000.0)
('Litecoin', 'LTC', '2023-10-02', 155.0, 11000000.0, 2200000.0)
('Ripple', 'XRP', '2023-10-02', 0.55, 26000000.0, 5500000.0)
('Bitcoin', 'BTC', '2023-10-03', 42000.0, 820000000.0, 32000000.0)
('Ethereum', 'ETH', '2023-10-03', 2100.0, 220000000.0, 17000000.0)
('Litecoin', 'LTC', '2023-10-03', 160.0, 12000000.0, 2400000.0)
('Ripple', 'XRP', '2023-10-03', 0.6, 27000000.0, 6000000.0)
('Bitcoin', 'BTC', '2023-10-04', 43000.0, 830000000.0, 33000000.0)
('Ethereum', 'ETH', '2023-10-04', 2150.0, 230000000.0, 18000000.0)
('Litecoin', 'LTC', '2023-10-04', 165.0, 13000000.0, 2600000.0)
('Ripple', 'XRP', '2023-10-

In [ ]:
# Reading the data into a DataFrame
conn = sqlite3.connect(db_path)
# Read the data into a DataFrame
df = pd.read_sql_query(query, conn)
# Close the connection
conn.close()
print(df.head())


       name symbol        date    price   market_cap  volume_24h
0   Bitcoin    BTC  2023-10-01  40000.0  800000000.0  30000000.0
1  Ethereum    ETH  2023-10-01   2000.5  200000000.0  15000000.0
2  Litecoin    LTC  2023-10-01    150.0   10000000.0   2000000.0
3    Ripple    XRP  2023-10-01      0.5   25000000.0   5000000.0
4   Bitcoin    BTC  2023-10-02  41000.0  810000000.0  31000000.0


In [17]:
# Calculate the daily average price, market cap for each coin
conn = sqlite3.connect(db_path)
avg_price_query = """
SELECT date, symbol, AVG(price) AS avg_price, AVG(market_cap) AS avg_market_cap
FROM prices
GROUP BY 1, 2
ORDER BY 1 DESC
;
"""
avg_prices = pd.read_sql_query(avg_price_query, conn)
conn.close()

# Add symbol as index
avg_prices.set_index('symbol', inplace=True)
avg_prices.head(10)

,date,avg_price,avg_market_cap
symbol,,,
BTC,2023-10-06,45000.00,850000000.0
ETH,2023-10-06,2250.00,250000000.0
LTC,2023-10-06,175.00,15000000.0
XRP,2023-10-06,0.75,30000000.0
BTC,2023-10-05,44000.00,840000000.0
ETH,2023-10-05,2200.00,240000000.0
LTC,2023-10-05,170.00,14000000.0
XRP,2023-10-05,0.70,29000000.0
BTC,2023-10-04,43000.00,830000000.0


In [ ]:
# Get coin with the highest total trade volume
conn = sqlite3.connect(db_path)
max_trade = pd.read_sql("""
SELECT
symbol, SUM(volume_24h) AS volume
FROM prices
ORDER BY volume DESC
LIMIT 1
""", conn)

max_trade

,symbol,volume
0,ETH,352500000.0
